# Single vs Multi-Source Split Modes

This tutorial compares **per_unit** (single-source) and **cross_unit** (multi-source) split modes. The split mode is determined by `get_split_mode()` and depends on whether `multisource_data_splitter` is set.

## Data flow: Single vs Multi

```
SINGLE (per_unit)                         MULTI (cross_unit)
─────────────────────                      ─────────────────────
ToyRaggedLoader                            MultiSourceToyLoader
  multisource_data_splitter = None           multisource_data_splitter = object()
  │                                         │
  │ load_data()                              │ load_data()
  │ get_data()                               │ get_data()
  ▼                                         ▼
get_split_mode() → "per_unit"              get_split_mode() → "cross_unit"
  │                                         │
  │ Each source split independently         │ Splits coordinated across sources
  │ (train/val/test per unit/source)        │ (joint train/val/test across sources)
```

## When to use which

| Mode | Use when | Example |
|------|----------|--------|
| **per_unit** | Single datasource, or each source has its own predefined splits | ToyRaggedLoader, PHMD, UNIBO |
| **cross_unit** | Multiple datasources that must be split jointly (e.g. train/val/test aligned across sources) | C-MAPSS DS02+DS03 combined |

In [ ]:
from picid.data.datasources.toy_example import ToyRaggedLoader
from picid.data.data_objects import SplitDatasetContainer


class MultiSourceToyLoader(ToyRaggedLoader):
    """Toy loader with multisource_data_splitter set to demonstrate cross_unit mode."""

    def __init__(self, **kwargs):
        kwargs.setdefault("multisource_data_splitter", object())
        super().__init__(**kwargs)

In [ ]:
# Single source: ToyRaggedLoader (no multisource_data_splitter)
single = ToyRaggedLoader(data_dir=".", data_name="toy", task_mode="anomaly_detection")
single.load_data()
assert single.get_split_mode() == "per_unit"
container_single = single.get_data()
assert isinstance(container_single, SplitDatasetContainer)
print(f"Single: split_mode = {single.get_split_mode()}")

In [ ]:
# Multi source: MultiSourceToyLoader (multisource_data_splitter=object())
multi = MultiSourceToyLoader(
    data_dir=".", data_name="toy_multi", task_mode="anomaly_detection"
)
multi.load_data()
assert multi.get_split_mode() == "cross_unit"
container_multi = multi.get_data()
assert isinstance(container_multi, SplitDatasetContainer)
print(f"Multi:  split_mode = {multi.get_split_mode()}")

In [ ]:
# Compare both
print(f"Single (ToyRaggedLoader):        split_mode = {single.get_split_mode()}")
print(f"Multi  (MultiSourceToyLoader):   split_mode = {multi.get_split_mode()}")
print("OK")